In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor



model = AutoModelForImageTextToText.from_pretrained(
    "LiquidAI/LFM3-VL-2B",
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

model_new = AutoModelForImageTextToText.from_pretrained(
    "out/lfm3-vl-2b-ft/final",
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

processor = AutoProcessor.from_pretrained(
    "LiquidAI/LFM3-VL-2B", trust_remote_code=True
)
tokenizer = processor.tokenizer

In [ ]:
import json
from PIL import Image

MAX_SIZE = 2048
dataset = open("dataset.jsonl").readlines()
prompt = json.loads(dataset[0])["conversations"][0]["value"]

image = "./dataset/eval/images/Kaufland-11-11-2026-17-11-2026-00_0.png"
image = Image.open(image)
image.thumbnail((MAX_SIZE, MAX_SIZE))

message = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": image,
            },
            {"type": "text", "text": prompt},
        ],
    }
]

inputs = processor.apply_chat_template(
    message,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)
inputs = inputs.to(model.device)

print(processor.decode(inputs["input_ids"][0]))


In [ ]:
from vlm_training.utils.bbox_viz import show_bboxes

# Inference: Generation of the output
with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        do_sample=True,
        temperature=0.1,
        top_k=50,
        repetition_penalty=1.1,
        max_new_tokens=16384,  # reasoning can be long
    )
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text[0])

 
show_bboxes(image, output_text[0])

In [ ]:
from vlm_training.utils.bbox_viz import show_bboxes

# Inference: Generation of the output
with torch.inference_mode():
    generated_ids = model_new.generate(
        **inputs,
        do_sample=True,
        temperature=0.1,
        top_k=50,
        repetition_penalty=None,
        max_new_tokens=16384,  # reasoning can be long
    )
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

print(output_text[0])

show_bboxes(image, output_text[0], coord_range=1000)


In [ ]:
import json
from vlm_training.utils.bbox_viz import compare_bboxes, parse_bboxes
 
img_name = "Kaufland-14-10-2026-20-10-2026-00_52.png"   # the file you're evaluating
row = next(r for r in map(json.loads, open("dataset.jsonl"))
           if img_name in r["images"][0])
gt = parse_bboxes(row["conversations"][1]["value"])
pred = parse_bboxes(output_text[0])
compare_bboxes(image, gt, pred)

In [ ]:
import json
from PIL import Image
 
row = next(r for r in map(json.loads, open("dataset.jsonl"))
           if "Kaufland-14-10-2026-20-10-2026-00_52.png" in r["images"][0])
img = Image.open(row["images"][0]).convert("RGB")
img.thumbnail((1024, 1024))                      # same as training
msg = [{"role": "user", "content": [
    {"type": "image", "image": img},
    {"type": "text", "text": row["conversations"][0]["value"]},  # includes <image>
]}]
inputs = processor.apply_chat_template(msg, tokenize=True, add_generation_prompt=True,
                                       return_dict=True, return_tensors="pt").to(model_new.device)
with torch.inference_mode():
    out = model_new.generate(**inputs, do_sample=False, max_new_tokens=2048)  # greedy, no rep penalty
txt = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(txt)

In [ ]:
def run(img):
    img = img.convert("RGB"); img.thumbnail((1024, 1024))
    msg = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": row["conversations"][0]["value"]}]}]
    inp = processor.apply_chat_template(msg, tokenize=True, add_generation_prompt=True,
                                        return_dict=True, return_tensors="pt").to(model_new.device)
    with torch.inference_mode():
        o = model_new.generate(**inp, do_sample=False, max_new_tokens=2048)
    return processor.decode(o[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
 
print(run(Image.open("./dataset/images/Akcni-letak-OD-PONDELI-19-10-21-10-2026-00_12.png")))
print(run(Image.open("./dataset/images/Kaufland-14-10-2026-20-10-2026-00_52.png")))